In [ ]:
import os
from glob import glob
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt


VIDEO = "V155"
VERSION = "1"
DATASET = "v155-obj-bg-masks"        # folder containing object frames, background frame and object masks
BG_FRAME = "00001.jpg"              # which background image to use

INPUT_ROOT  = f"/kaggle/input/{DATASET}"
WORK_ROOT   = "/kaggle/working"

BACKGROUND_PATH = os.path.join(INPUT_ROOT, "bg", "bg", BG_FRAME)
FRAMES_DIR  = os.path.join(INPUT_ROOT, "obj", "obj")
MASKS_DIR   = os.path.join(INPUT_ROOT, "obj_masks", "obj_masks")

OUT_MASK_DIR    = f"shadow_masks_{VIDEO}_v{VERSION}"
OUT_COLOR_DIR   = f"shadow_color_{VIDEO}_v{VERSION}"
OUT_OVERLAY_DIR = f"shadow_overlay_{VIDEO}_v{VERSION}"

shadow_zip_path  = os.path.join(WORK_ROOT, f"shadow_color_{VIDEO}_v{VERSION}.zip")
mask_zip_path    = os.path.join(WORK_ROOT, f"shadow_masks_{VIDEO}_v{VERSION}.zip")
overlay_zip_path = os.path.join(WORK_ROOT, f"shadow_overlay_{VIDEO}_v{VERSION}.zip")

# Detection thresholds
T_DARK_MAX = 0.95
T_DARK_MIN = 0.0
MIN_AREA = 50
EPS = 1e-3

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def list_frames_sorted(folder):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    files = [f for f in glob(os.path.join(folder, "*")) if f.lower().endswith(exts)]
    files.sort()
    return files

def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(folder_path):
            for fname in files:
                full_path = os.path.join(root, fname)
                rel_path = os.path.relpath(full_path, folder_path)
                zf.write(full_path, arcname=rel_path)

# Load background
bg_bgr = cv2.imread(BACKGROUND_PATH, cv2.IMREAD_COLOR)
if bg_bgr is None:
    raise FileNotFoundError(f"Could not read background image: {BACKGROUND_PATH}")

h, w, _ = bg_bgr.shape
print(f"Background loaded: {BACKGROUND_PATH}")
print(f"Size: {w} x {h}")

# Convert background to YCrCb once
bg_ycrcb = cv2.cvtColor(bg_bgr, cv2.COLOR_BGR2YCrCb)
bg_Y, bg_Cr, bg_Cb = cv2.split(bg_ycrcb)
bg_Y = bg_Y.astype(np.float32)


# Display background WITH SCALE + LAST PIXEL VALUES
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(bg_bgr, cv2.COLOR_BGR2RGB))

plt.title(
    f"Background: {BG_FRAME} | "
    f"X: 0 → {w-1}, Y: 0 → {h-1}",
    fontsize=12
)

# Choose tick spacing
x_step = max(50, w // 20)
y_step = max(50, h // 20)

# Build ticks and FORCE last pixel to appear
x_ticks = list(np.arange(0, w, x_step))
y_ticks = list(np.arange(0, h, y_step))

if x_ticks[-1] != w - 1:
    x_ticks.append(w - 1)
if y_ticks[-1] != h - 1:
    y_ticks.append(h - 1)

plt.xticks(x_ticks)
plt.yticks(y_ticks)

plt.xlabel("x (pixels)")
plt.ylabel("y (pixels)")

plt.grid(color="yellow", linestyle="--", linewidth=0.5, alpha=0.6)
plt.axis("on")
plt.show()


In [ ]:

# ENTER YOUR BOUNDING BOX HERE
# Format: x1, y1, x2, y2  (x2/y2 are exclusive or inclusive doesn't matter much. we clamp it.
x1, y1, x2, y2 = 0, 0, 847, 479   # <-- CHANGE THIS

# Safety clamp
x1 = max(0, min(x1, w-1)); x2 = max(1, min(x2, w))
y1 = max(0, min(y1, h-1)); y2 = max(1, min(y2, h))
if x2 <= x1 or y2 <= y1:
    raise ValueError("Invalid bounding box. Make sure x2>x1 and y2>y1.")

print(f"Using ROI bbox: (x1={x1}, y1={y1}, x2={x2}, y2={y2})")

# Prepare output dirs
ensure_dir(OUT_MASK_DIR)
ensure_dir(OUT_COLOR_DIR)
ensure_dir(OUT_OVERLAY_DIR)

frame_paths = list_frames_sorted(FRAMES_DIR)
print(f"Found {len(frame_paths)} frames.")

# Pre-create kernels once
kernel_dil = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

# Crop background channels to ROI once
bg_Y_roi  = bg_Y[y1:y2, x1:x2]
bg_Cr_roi = bg_Cr[y1:y2, x1:x2]
bg_Cb_roi = bg_Cb[y1:y2, x1:x2]

for frame_path in frame_paths:
    fname = os.path.basename(frame_path)
    name, ext = os.path.splitext(fname)

    try:
        frame_idx = int(name)
    except ValueError:
        print(f"[WARN] Unexpected frame filename format: {fname}, skipping.")
        continue

    mask_name = f"frame_{frame_idx:04d}.png"
    mask_path = os.path.join(MASKS_DIR, mask_name)

    if not os.path.exists(mask_path):
        print(f"[INFO] No mask for frame {fname} (expected {mask_name}), skipping.")
        continue

    frame_bgr = cv2.imread(frame_path, cv2.IMREAD_COLOR)
    obj_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if frame_bgr is None or obj_mask is None:
        print(f"[WARN] Could not read frame/mask for {fname}, skipping.")
        continue
    if frame_bgr.shape[:2] != (h, w):
        print(f"[WARN] Size mismatch for {frame_path}, skipping.")
        continue

    # ROI crop for speed
    frame_roi = frame_bgr[y1:y2, x1:x2]
    obj_mask_roi = obj_mask[y1:y2, x1:x2]

    # Convert ROI to YCrCb
    frame_ycrcb_roi = cv2.cvtColor(frame_roi, cv2.COLOR_BGR2YCrCb)
    fr_Y_roi, fr_Cr_roi, fr_Cb_roi = cv2.split(frame_ycrcb_roi)
    fr_Y_roi = fr_Y_roi.astype(np.float32)

    # Object mask binarize + dilate (ROI)
    _, obj_mask_bin_roi = cv2.threshold(obj_mask_roi, 128, 1, cv2.THRESH_BINARY)
    obj_mask_dil_roi = cv2.dilate(obj_mask_bin_roi.astype(np.uint8), kernel_dil, iterations=1)
    non_obj_roi = (obj_mask_dil_roi == 0)

    # Brightness ratio in ROI
    ratio_roi = fr_Y_roi / (bg_Y_roi + EPS)
    ratio_roi = cv2.GaussianBlur(ratio_roi, (5, 5), 0)
    cond_dark_roi = (ratio_roi < T_DARK_MAX) & (ratio_roi > T_DARK_MIN)

    shadow_candidate_roi = cond_dark_roi & non_obj_roi

    # Build full-size shadow mask, but fill only ROI
    shadow_mask = np.zeros((h, w), dtype=np.uint8)
    shadow_mask_roi = np.zeros((y2-y1, x2-x1), dtype=np.uint8)
    shadow_mask_roi[shadow_candidate_roi] = 255

    # Morph cleanup only inside ROI
    shadow_mask_roi = cv2.morphologyEx(shadow_mask_roi, cv2.MORPH_OPEN, kernel_small, iterations=1)
    shadow_mask_roi = cv2.morphologyEx(shadow_mask_roi, cv2.MORPH_CLOSE, kernel_small, iterations=1)

    # Small component filtering only in ROI
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(shadow_mask_roi, connectivity=8)
    cleaned_roi = np.zeros_like(shadow_mask_roi)
    for lab in range(1, num_labels):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= MIN_AREA:
            cleaned_roi[labels == lab] = 255
    shadow_mask_roi = cleaned_roi

    # Put ROI result into full mask
    shadow_mask[y1:y2, x1:x2] = shadow_mask_roi

    # Save binary shadow mask
    out_mask_name = f"{name}_shadow_mask.png"
    mask_out_path = os.path.join(OUT_MASK_DIR, out_mask_name)
    cv2.imwrite(mask_out_path, shadow_mask)

    # Shadow-only color image
    shadow_color = np.zeros_like(frame_bgr)
    shadow_pixels = (shadow_mask == 255)
    shadow_color[shadow_pixels] = frame_bgr[shadow_pixels]
    out_color_name = f"{name}_shadow_color.png"
    color_out_path = os.path.join(OUT_COLOR_DIR, out_color_name)
    cv2.imwrite(color_out_path, shadow_color)

    # Overlay
    overlay = frame_bgr.copy()
    overlay[shadow_pixels] = (0, 0, 255)
    alpha = 0.5
    shadow_overlay = cv2.addWeighted(overlay, alpha, frame_bgr, 1 - alpha, 0)
    out_overlay_name = f"{name}_shadow_overlay.png"
    overlay_out_path = os.path.join(OUT_OVERLAY_DIR, out_overlay_name)
    cv2.imwrite(overlay_out_path, shadow_overlay)

    print(f"Processed {fname} -> {out_mask_name}")

print("All done.")

zip_folder(OUT_COLOR_DIR, shadow_zip_path)
zip_folder(OUT_MASK_DIR, mask_zip_path)
zip_folder(OUT_OVERLAY_DIR, overlay_zip_path)

print("Saved zip files:")
print(shadow_zip_path)
print(mask_zip_path)
print(overlay_zip_path)
